# Adversarial Debate — CrewAI

**Pattern:** Proposer → Critic → Judge with `DebateVerdict` output

```
proposer ──FOR──▶ critic ──AGAINST──▶ judge ──▶ DebateVerdict
                 context=[propose_task] context=[propose+critique]
```

Agents have opposing backstories. `output_pydantic=DebateVerdict` gives scored output.

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0.3)
print("✓ Ready")

In [ ]:
class DebateVerdict(BaseModel):
    proposer_score: int = Field(ge=1, le=10)
    critic_score: int = Field(ge=1, le=10)
    stronger_side: str
    key_insight: str
    final_conclusion: str

proposer = Agent(role="Travel Advocate", goal="Make strongest case FOR the claim.",
    backstory="Passionate travel writer who defends bold opinions.", tools=[], llm=gemini, verbose=False)
critic   = Agent(role="Travel Skeptic", goal="Find weaknesses in the claim.",
    backstory="Seasoned traveler who exposes overhyped destinations.", tools=[], llm=gemini, verbose=False)
judge    = Agent(role="Impartial Judge", goal="Deliver fair scored verdict.",
    backstory="Neutral travel analyst who evaluates arguments on merit.", tools=[], llm=gemini, verbose=False)
print("3 debate agents ready")

In [ ]:
claim = "Tokyo is the best travel destination for a one-week trip"

t1 = Task(description=f"Argue FOR: '{claim}'. 3 specific arguments, 150 words.",
    expected_output="Persuasive FOR argument.", agent=proposer)
t2 = Task(description=f"Argue AGAINST: '{claim}'. Challenge the proposer. 150 words.",
    expected_output="Rigorous AGAINST argument.", agent=critic, context=[t1])
t3 = Task(description=f"Judge the debate about '{claim}'. Score both sides 1-10. Fill DebateVerdict.",
    expected_output="Complete DebateVerdict.", agent=judge, context=[t1,t2], output_pydantic=DebateVerdict)

result = Crew(agents=[proposer,critic,judge], tasks=[t1,t2,t3], process=Process.sequential, verbose=False).kickoff()
v: DebateVerdict = result.pydantic
if v:
    print(f"FOR score:     {v.proposer_score}/10")
    print(f"AGAINST score: {v.critic_score}/10")
    print(f"Stronger side: {v.stronger_side}")
    print(f"Key insight:   {v.key_insight}")
    print(f"\nConclusion:\n{v.final_conclusion}")
else: print(result.raw)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Opposing backstories | CrewAI personas go deeper than just prompt instructions |
| `output_pydantic=DebateVerdict` | Structured verdict — scores are integers, not text |
| Judge has `context=[t1,t2]` | Judge sees the FULL debate, not just the latest message |

### Next: [ADK Debate →](../ADK/debate.ipynb)